In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/semeval-subtask-3/task3-df.csv")
print(df.shape)
df.head()

(12291, 4)


,article_id,text_id,label,text
0,813452859,1,NaN,EU Profits From Trading With UK While London L...
1,813452859,3,NaN,With the Parliamentary vote on British Prime M...
2,813452859,4,NaN,But is there any chance that May's deal will m...
3,813452859,5,NaN,Sputnik spoke with political campaigner Michae...
4,813452859,6,NaN,Sputnik: Does Theresa May have any chance of g...


In [3]:
df['label'].value_counts()

label
Loaded_Language                                                                         1062
Name_Calling-Labeling                                                                    417
Doubt                                                                                    309
Repetition                                                                               263
Loaded_Language,Name_Calling-Labeling                                                    218
                                                                                        ... 
Appeal_to_Hypocrisy,Repetition                                                             1
Doubt,Loaded_Language,Slogans                                                              1
Doubt,Flag_Waving,Loaded_Language,Repetition                                               1
Appeal_to_Hypocrisy,Doubt,Guilt_by_Association,Loaded_Language,Name_Calling-Labeling       1
Doubt,Exaggeration-Minimisation,Repetition                      

In [4]:
df['label_list'] = df['label'].apply(lambda x: x.split(',') if pd.notna(x) else [])
df.tail()

,article_id,text_id,label,text,label_list
12286,999001970,9,NaN,"Patel pushed back on the officials’ remarks, a...",[]
12287,999001970,10,NaN,The real world? This is Columbia.,[]
12288,999001970,11,NaN,"For Sofia Jao, BC ‘22, problems with the perfo...",[]
12289,999001970,12,NaN,Patel is 32.,[]
12290,999001970,13,Exaggeration-Minimisation,"I'm sure Patel felt very, like, accepted.",[Exaggeration-Minimisation]


In [220]:
new_df = df[df['label'].isin(["Loaded_Language", "Name_Calling-Labeling", "Doubt", "Repetition"])]
print(new_df.shape)
new_df.head()

(2051, 5)


,article_id,text_id,label,text,label_list
19,813494037,1,Loaded_Language,Sadiq Khan Slammed for Pro-EU 'Message of Supp...,[Loaded_Language]
20,813494037,3,Loaded_Language,The spectacular fireworks that lit up the Lond...,[Loaded_Language]
22,813494037,5,Repetition,The 135-metre-high London Eye was lit up in bl...,[Repetition]
25,813494037,8,Repetition,I'm proud that tonight we will welcome in the ...,[Repetition]
29,813494037,12,Loaded_Language,This man has no shame.,[Loaded_Language]


In [221]:
new_df['label'].value_counts()

label
Loaded_Language          1062
Name_Calling-Labeling     417
Doubt                     309
Repetition                263
Name: count, dtype: int64

In [222]:
new_df1 = new_df[['text', 'label']]
new_df1.head()

,text,label
19,Sadiq Khan Slammed for Pro-EU 'Message of Supp...,Loaded_Language
20,The spectacular fireworks that lit up the Lond...,Loaded_Language
22,The 135-metre-high London Eye was lit up in bl...,Repetition
25,I'm proud that tonight we will welcome in the ...,Repetition
29,This man has no shame.,Loaded_Language


In [223]:
label_mapping = {
    "Loaded_Language": 0,
    "Name_Calling-Labeling": 1,
    "Doubt": 2,
    "Repetition": 3
}

new_df1['label'] = new_df1['label'].map(label_mapping)
print(new_df1[new_df1['label'].isna()])
new_df1['label'].astype(int)
new_df1.head()

Empty DataFrame
Columns: [text, label]
Index: []


,text,label
19,Sadiq Khan Slammed for Pro-EU 'Message of Supp...,0
20,The spectacular fireworks that lit up the Lond...,0
22,The 135-metre-high London Eye was lit up in bl...,3
25,I'm proud that tonight we will welcome in the ...,3
29,This man has no shame.,0


## Tokenize

In [224]:
%pip install nltk
from IPython.display import clear_output
clear_output()

from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/erlo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [225]:
def split_tokens(row):
    row['tokens'] = word_tokenize(row['text'].lower())
    return row
new_df1 = new_df1.apply(split_tokens, axis=1)
new_df1.head()

,text,label,tokens
19,Sadiq Khan Slammed for Pro-EU 'Message of Supp...,0,"[sadiq, khan, slammed, for, pro-eu, 'message, ..."
20,The spectacular fireworks that lit up the Lond...,0,"[the, spectacular, fireworks, that, lit, up, t..."
22,The 135-metre-high London Eye was lit up in bl...,3,"[the, 135-metre-high, london, eye, was, lit, u..."
25,I'm proud that tonight we will welcome in the ...,3,"[i, 'm, proud, that, tonight, we, will, welcom..."
29,This man has no shame.,0,"[this, man, has, no, shame, .]"


In [226]:
all_tokens = [token for tokens in new_df1['tokens'] for token in tokens]
unique_tokens = set(all_tokens)
print(f"Total unique tokens: {len(unique_tokens)}")

from collections import Counter
token_counts = Counter(all_tokens)
token_counts.most_common(20)

Total unique tokens: 11482


[('the', 5700),
 (',', 4513),
 ('.', 3334),
 ('of', 2653),
 ('to', 2504),
 ('and', 2027),
 ('a', 1819),
 ('in', 1687),
 ('that', 1420),
 ('’', 1136),
 ('is', 988),
 ('“', 871),
 ('”', 861),
 ('for', 763),
 ('s', 743),
 ('was', 720),
 ('it', 691),
 ('on', 627),
 ('he', 580),
 ('as', 561)]

In [227]:
vocab = [token for token, count in token_counts.items() if count >= 2]

VOCAB_SIZE = len(vocab)

print(f"Vocabulary size: {VOCAB_SIZE}")

Vocabulary size: 5713


In [228]:
token2id = {token: idx for idx, token in enumerate(vocab)}

In [229]:
def convert_tokens(tokens):
    result = []
    for token in tokens:
        result.append(token2id.get(token, 0))

    return result

def tokens2ids(row):
    row['tokens_id'] = convert_tokens(row['tokens'])
    return row

new_df1 = new_df1.apply(tokens2ids, axis=1)
new_df1.head()

,text,label,tokens,tokens_id
19,Sadiq Khan Slammed for Pro-EU 'Message of Supp...,0,"[sadiq, khan, slammed, for, pro-eu, 'message, ...","[0, 1, 2, 3, 4, 0, 5, 6, 7, 8, 0, 9]"
20,The spectacular fireworks that lit up the Lond...,0,"[the, spectacular, fireworks, that, lit, up, t...","[10, 11, 12, 13, 14, 15, 10, 16, 17, 18, 19, 2..."
22,The 135-metre-high London Eye was lit up in bl...,3,"[the, 135-metre-high, london, eye, was, lit, u...","[10, 0, 16, 41, 42, 14, 15, 43, 44, 45, 35, 0,..."
25,I'm proud that tonight we will welcome in the ...,3,"[i, 'm, proud, that, tonight, we, will, welcom...","[61, 62, 0, 13, 63, 64, 65, 66, 43, 10, 67, 68..."
29,This man has no shame.,0,"[this, man, has, no, shame, .]","[70, 71, 72, 73, 74, 40]"


In [230]:
# separate a new df for testing (20%)

df_test = new_df1.sample(frac=0.2, random_state=42)
df_train = new_df1.drop(df_test.index)

## DataLoader

In [231]:
import torch
from torch.utils.data import Dataset

class SE3Dataset(Dataset):
    def __init__(self, dataset):
        self.data = dataset['tokens_id'].tolist()
        self.labels = dataset['label'].tolist()
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, index):
        x = torch.tensor(self.data[index], dtype=torch.long)
        y = torch.tensor(int(self.labels[index]), dtype=torch.long)
        return x, y

In [232]:
dataset = SE3Dataset(df_train)
print(f"Dataset size: {len(dataset)}")

Dataset size: 1641


In [233]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    xs, ys = zip(*batch)
    xs_padded = pad_sequence(xs, batch_first=True, padding_value=0)
    ys = torch.stack(ys)
    return xs_padded, ys

dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [234]:
from torch import nn

EMBED_LEN = 128
HIDDEN_DIM = 32
N_LAYERS = 1
N_CLASSES = 4

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_len, hidden_dim, n_layers, n_classes):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_len)
        self.rnn = nn.RNN(embed_len, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        last_hidden = hidden[-1]
        out = self.fc(last_hidden)
        return out

In [235]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [236]:
model = RNNClassifier(VOCAB_SIZE, EMBED_LEN, HIDDEN_DIM, N_LAYERS, N_CLASSES).to(device)

In [237]:
LR = 1e-3
EPOCHS = 5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [238]:
from tqdm.notebook import tqdm

def train(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in tqdm(dataloader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = loss_fn(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def test(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in tqdm(dataloader):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch).squeeze()
            loss = loss_fn(outputs, y_batch)
            total_loss += loss.item()
    return total_loss / len(dataloader)

In [239]:
dataset_test = SE3Dataset(df_test)
dataloader_test = DataLoader(dataset_test, batch_size=32, collate_fn=collate_fn)

In [240]:
from datetime import datetime

time_now = datetime.now().strftime("%Y-%m-%d_%H-%Mh")
time_now

'2026-03-18_23-15h'

In [241]:
train_losses = []
test_losses = []

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}\n------------")

    print('Train')
    train_losses.append(train(model, dataloader, loss_fn, optimizer, device))

    print('Test')
    test_losses.append(test(model, dataloader, loss_fn, device))

    torch.save(model.state_dict(), f'models/{time_now}_rnn_se3_{epoch+1}.pth')

Epoch 1
------------
Train


  0%|          | 0/52 [00:00<?, ?it/s]

Test


  0%|          | 0/52 [00:00<?, ?it/s]

Epoch 2
------------
Train


...

  0%|          | 0/52 [00:00<?, ?it/s]

Epoch 5
------------
Train


  0%|          | 0/52 [00:00<?, ?it/s]

Test


  0%|          | 0/52 [00:00<?, ?it/s]

In [243]:
# Mapeamento reverso para decodificar predições
id2label = {v: k for k, v in label_mapping.items()}

# Exemplos de texto para testar
examples = {
    "Loaded_Language": "This is the most amazing and incredible product ever created in human history!",
    "Name_Calling-Labeling": "These politicians are just stupid idiots who don't understand anything.",
    "Doubt": "Maybe this could be true, but I'm not entirely sure about it.",
    "Repetition": "We need change, change, change! Real change, real change, real change!"
}

# Função para fazer predição
def predict(text):
    tokens = word_tokenize(text.lower())
    token_ids = convert_tokens(tokens)
    
    # Converter para tensor e adicionar batch dimension
    input_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_class].item()
    
    return predicted_class, confidence

# Testar cada exemplo
print("=" * 80)
print("INFERÊNCIA - CLASSIFICAÇÃO DE PROPAGANDA")
print("=" * 80)

for label_name, text in examples.items():
    predicted_id, confidence = predict(text)
    predicted_label = id2label[predicted_id]
    
    print(f"\nTexto: {text}")
    print(f"Label esperado: {label_name}")
    print(f"Predição: {predicted_label}")
    print(f"Confiança: {confidence:.4f}")
    print("-" * 80)

INFERÊNCIA - CLASSIFICAÇÃO DE PROPAGANDA

Texto: This is the most amazing and incredible product ever created in human history!
Label esperado: Loaded_Language
Predição: Loaded_Language
Confiança: 0.5072
--------------------------------------------------------------------------------

Texto: These politicians are just stupid idiots who don't understand anything.
Label esperado: Name_Calling-Labeling
Predição: Loaded_Language
Confiança: 0.5420
--------------------------------------------------------------------------------

Texto: Maybe this could be true, but I'm not entirely sure about it.
Label esperado: Doubt
Predição: Loaded_Language
Confiança: 0.5538
--------------------------------------------------------------------------------

Texto: We need change, change, change! Real change, real change, real change!
Label esperado: Repetition
Predição: Loaded_Language
Confiança: 0.5396
--------------------------------------------------------------------------------
